# 🚀 HGGAN_test: All-in-One Kaggle Workspace Controller

Welcome to the unified training workspace for **HGGAN_test**. This notebook simplifies setup, synchronization, dataset linkage, concurrent experiment execution, process management, and Git tracking across all four repository branches:

1. **`main`**: The core production branch containing primary model updates and allographic upgrades.
2. **`dev`**: The active experimental playground for style and latent loss modifications.
3. **`classic_optimized`**: The baseline classical HGGAN codebase optimized for fast iterations.
4. **`alpha`**: Global-only style ablation matched to MAIN training settings.

---

## 📦 Step 1: Environment & Dependency Setup
Installs PyTorch and other baseline dependencies compatible with HGGAN execution.


In [ ]:
!pip install --no-cache-dir torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu118
!pip install --no-cache-dir numpy==2.2.6 scipy==1.15.3 scikit-learn==1.7.2 scikit-image==0.25.2 h5py==3.14.0 Pillow==12.2.0 matplotlib==3.10.0 opencv-python-headless==4.13.0.92 munch==4.0.0 Distance==0.1.3 tqdm==4.67.3 PyYAML==6.0.3 wandb==0.27.0 timm==1.0.19 editdistance==0.8.1 transformers==5.9.0 sentencepiece==0.2.1 huggingface-hub==1.16.4

In [ ]:
import os
import torch

# ========================================================
# MASTER CONFIGURATION: Set the mother directory path
# ========================================================
MASTER_PATH = "/kaggle/working"

# ── Secrets Setup (Kaggle Secrets with Local Fallback) ──
WANDB_KEY = ""
HF_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    try: WANDB_KEY = user_secrets.get_secret("WANDB_KEY")
    except Exception: pass
    try: HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    except Exception: pass
except Exception:
    pass

if not WANDB_KEY:
    WANDB_KEY = os.environ.get("WANDB_API_KEY", "")
    wandb_txt = os.path.join(MASTER_PATH, "wandb_key.txt")
    if not WANDB_KEY and os.path.exists(wandb_txt):
        try:
            with open(wandb_txt, "r") as f:
                WANDB_KEY = f.read().strip()
        except Exception: pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", os.environ.get("HUGGING_FACE_HUB_TOKEN", ""))

if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    wandb_txt = os.path.join(MASTER_PATH, "wandb_key.txt")
    try:
        with open(wandb_txt, "w") as f:
            f.write(WANDB_KEY)
        print("✅ WandB key loaded and saved to wandb_key.txt")
    except Exception: pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("✅ HuggingFace token set in environment variables")

print('✅ PyTorch version:', torch.__version__)
print('✅ CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✅ Device Name:', torch.cuda.get_device_name(0))

In [ ]:
# Independent OCR weights are cached once and shared by MAIN/ALPHA.
# No GPU is used here. Disable if not running GAN OCR evaluation.
PREPARE_OCR_EVALUATOR = True
if PREPARE_OCR_EVALUATOR:
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(
            repo_id="microsoft/trocr-small-handwritten",
            revision="b4648cfa171985a6745f37ddd637e98c0da958ac",
            allow_patterns=["*.json", "pytorch_model.bin", "sentencepiece.bpe.model"],
            max_workers=2,
        )
        print("Independent OCR evaluator cached: TrOCR-small (~246 MB).")
    except Exception as error:
        print(f"OCR cache unavailable: {error}. GAN can run with CER/WER disabled; no teacher fallback.")


## 🔄 Step 2: Clone & Sync All Branches
Clones all branches of your GitHub repository `https://github.com/rquq/HGGAN_test.git` into independent subdirectories under `/kaggle/working/HGGAN_test/`. If they already exist, they will be hard-reset to their latest remote versions.

In [ ]:
import os
import shutil

repo_url = "https://github.com/rquq/HGGAN_test.git"
base_dir = os.path.join(MASTER_PATH, "HGGAN_test")
branches = ["main", "dev", "classic_optimized", "alpha"]

os.makedirs(base_dir, exist_ok=True)

# Set Git global configs for smooth setup
!git config --global user.email "williamwrost617@gmail.com"
!git config --global user.name "rquq"

for branch in branches:
    branch_dir = os.path.join(base_dir, branch)
    if os.path.exists(branch_dir):
        print(f"🔄 Resetting and updating local folder for branch: {branch}...")
        !git -C {branch_dir} fetch origin
        !git -C {branch_dir} reset --hard origin/{branch}
    else:
        print(f"📥 Cloning branch: {branch} from remote...")
        !git clone -b {branch} {repo_url} {branch_dir}

## 💾 Step 3: Centralized IAM Dataset Manager (x32 & x64)
Set `IAM_HEIGHTS` to `[32, 64]` (to download both resolutions simultaneously), or `[32]` or `[64]`. 
The notebook downloads the requested IAM release files into the shared `iam/` folder, then automatically links them into every branch (`main`, `dev`, `classic_optimized`, `alpha`) under standard and compatibility filenames (`iam_train_*.hdf5`, `trnvalset_words*.hdf5`, and `trnvalset_words*_OrgSz.hdf5`).

💡 **Multi-GPU Flexibility**: Downloading both resolutions allows you to concurrently run a 32x model on GPU 0 and a 64x model on GPU 1 without dataset path conflicts.

In [ ]:
import os
import shutil
from urllib.request import urlretrieve

# ========================================================
# DATASET CONFIGURATION: Choose resolutions to download & link
# Options: [32, 64] (both resolutions), or [32], or [64]
# ========================================================
IAM_HEIGHTS = [32, 64]  # Supports both resolutions simultaneously for flexible multi-GPU testing!

IAM_RELEASES = {
    32: {
        'train': ('iam_train_32.hdf5', 'https://github.com/rquq/HGGAN_test/releases/download/data/iam_train_32.hdf5'),
        'test': ('iam_test_32.hdf5', 'https://github.com/rquq/HGGAN_test/releases/download/data/iam_test_32.hdf5'),
    },
    64: {
        'train': ('iam_train_64.hdf5', 'https://github.com/rquq/HGGAN_test/releases/download/data/iam_train_64.hdf5'),
        'test': ('iam_test_64.hdf5', 'https://github.com/rquq/HGGAN_test/releases/download/data/iam_test_64.hdf5'),
    },
}

dataset_dir = os.path.join(MASTER_PATH, 'iam')
os.makedirs(dataset_dir, exist_ok=True)

def download_if_missing(filename, url):
    destination = os.path.join(dataset_dir, filename)
    if os.path.isfile(destination) and os.path.getsize(destination) > 0:
        print(f'✅ Using existing {filename} ({os.path.getsize(destination) / (1024*1024):.1f} MB)')
        return destination
    partial = destination + '.part'
    if os.path.lexists(partial):
        os.remove(partial)
    print(f'⬇️ Downloading {filename} from release...')
    urlretrieve(url, partial)
    os.replace(partial, destination)
    print(f'✅ Finished downloading {filename}')
    return destination

def replace_link(source, destination):
    if os.path.lexists(destination):
        os.remove(destination)
    try:
        os.symlink(source, destination)
    except Exception:
        shutil.copy(source, destination)

downloaded_paths = {}
for h in IAM_HEIGHTS:
    if h in IAM_RELEASES:
        tr_name, tr_url = IAM_RELEASES[h]['train']
        te_name, te_url = IAM_RELEASES[h]['test']
        tr_path = download_if_missing(tr_name, tr_url)
        te_path = download_if_missing(te_name, te_url)
        downloaded_paths[h] = {'train': tr_path, 'test': te_path}
    else:
        print(f'⚠️ Unsupported resolution height: {h}. Supported: 32, 64')

# Establish comprehensive symlinks into all branch data/iam directories
for branch in branches:
    branch_data_dir = os.path.join(base_dir, branch, 'data', 'iam')
    os.makedirs(branch_data_dir, exist_ok=True)
    for h, paths in downloaded_paths.items():
        tr_path, te_path = paths['train'], paths['test']
        
        # Standard filenames & OrgSz compatibility aliases
        train_aliases = [
            f'iam_train_{h}.hdf5',
            f'trnvalset_words{h}.hdf5',
            f'trnvalset_words{h}_OrgSz.hdf5'
        ]
        test_aliases = [
            f'iam_test_{h}.hdf5',
            f'testset_words{h}.hdf5',
            f'testset_words{h}_OrgSz.hdf5'
        ]
        
        for alias in train_aliases:
            replace_link(tr_path, os.path.join(branch_data_dir, alias))
        for alias in test_aliases:
            replace_link(te_path, os.path.join(branch_data_dir, alias))
        print(f'🔗 Linked IAM x{h} (train + test + aliases) to HGGAN_test/{branch}/data/iam/')

print(f'\n🎉 All requested datasets {IAM_HEIGHTS} are downloaded and linked across all branches!')


## 📥 Step 3.5: Resume Checkpoint Linker (Optional)
If you want to resume training from a saved checkpoint located in a Kaggle input dataset (e.g., `/kaggle/input/datasets/h0m3l4nd3r/...`), specify the paths below. This automatically symlinks the checkpoints into the respective branches and dynamically updates their config files (`gan_iam.yml` and `gan_iam_local.yml`) to point to these checkpoints when the code is run.

In [ ]:
# ========================================================
# OPTIONAL: Resume / Pretrained Checkpoint Linker
# ========================================================
# Map direct checkpoint file paths (e.g. from Kaggle input /kaggle/input/datasets/...)
# to their target branches. Set a branch to "none" to start from scratch or clear checkpoint.
CHECKPOINT_PATHS = {
    "main": "none",
    "dev": "none",
    "classic_optimized": "none",
    "alpha": "none"
}

import os
import shutil
import glob

def update_config_checkpoint(config_path, checkpoint_val):
    if not os.path.exists(config_path):
        return False
    
    with open(config_path, "r") as f:
        lines = f.readlines()
    
    updated = False
    new_lines = []
    for line in lines:
        if line.strip().startswith("pretrained_ckpt:"):
            indent = line[:line.index("pretrained_ckpt:")]
            new_lines.append(f"{indent}pretrained_ckpt: '{checkpoint_val}'\n")
            updated = True
        else:
            new_lines.append(line)
            
    if updated:
        with open(config_path, "w") as f:
            f.writelines(new_lines)
    return updated

active_links = False
for branch, path in CHECKPOINT_PATHS.items():
    branch_dir = os.path.join(base_dir, branch)
    configs_to_update = [
        os.path.join(branch_dir, "configs", "gan_iam_64.yml"),
        os.path.join(branch_dir, "configs", "gan_iam_32.yml"),
        os.path.join(branch_dir, "configs", "gan_iam_local.yml")
    ]
    
    if path == "none":
        for config_path in configs_to_update:
            if update_config_checkpoint(config_path, ""):
                print(f"🧹 Cleared 'pretrained_ckpt' in {os.path.relpath(config_path, base_dir)}")
        continue
        
    if not os.path.exists(path):
        print(f"⚠️ Checkpoint path not found: {path} - Skipping branch '{branch}' link.")
        continue
        
    active_links = True
    checkpoint_name = os.path.basename(path)
    print(f"🔗 Branch '{branch}': Linking resume checkpoint from: {path}")
    
    branch_pretrained_dir = os.path.join(branch_dir, "pretrained")
    os.makedirs(branch_pretrained_dir, exist_ok=True)
    
    target_link = os.path.join(branch_pretrained_dir, checkpoint_name)
    if os.path.exists(target_link) or os.path.islink(target_link):
        os.remove(target_link)
    try:
        os.symlink(path, target_link)
        print(f"   ✅ Symlinked: HGGAN_test/{branch}/pretrained/{checkpoint_name} -> {path}")
    except Exception:
        shutil.copy(path, target_link)
        print(f"   ✅ Linked (copied fallback): HGGAN_test/{branch}/pretrained/{checkpoint_name}")

    ckpt_relative_path = f"./pretrained/{checkpoint_name}"
    for config_path in configs_to_update:
        if update_config_checkpoint(config_path, ckpt_relative_path):
            print(f"   📝 Updated 'pretrained_ckpt' to '{ckpt_relative_path}' in {os.path.relpath(config_path, base_dir)}")

if not active_links:
    print("ℹ️ No active checkpoint links configured (all set to 'none' or missing). All branches will start training from scratch.")

## 🏋️ Step 4: Launch training jobs
Run the OCR teacher cell first. It launches Classic OCR32-source on GPU0 and OCR64-source on GPU1. After both RESULTS tables appear, switch the two configs to `wid_iam_32.yml` and `wid_iam_64.yml` and run the same cell again for W+B. The optional GAN cell is disabled by default.

The monitor exits after every selected job finishes and prints each run's RESULTS table path.


In [ ]:
#configs

RUN_GAN_JOBS = False
BG_BRANCH_GPU0 = "main"
BG_CONFIG_GPU0 = "configs/gan_iam_32.yml"
BG_BRANCH_GPU1 = "alpha"
BG_CONFIG_GPU1 = "configs/gan_iam_32.yml"


# Run OCR first, wait for RESULTS, then switch the two configs to wid_iam_32.yml / wid_iam_64.yml.
RUN_TEACHER_PRETRAIN = False
GPU0_BRANCH = "classic_optimized"
GPU0_CONFIG = "configs/ocr_iam_32.yml"
GPU1_BRANCH = "classic_optimized"
GPU1_CONFIG = "configs/ocr_iam_64.yml"


In [ ]:
#time limit to stop the monitor print cell (in hrs) so kaggle save version auto stop session, for usage control
TIME_LIMIT = 5.5


In [ ]:
# ========================================================
# CONFIGURATION: Choose branches and configs to run concurrently
# ========================================================

import glob
import os
import shutil
import shlex
import sys
import time


def copy_baseline_best_to_run_folder(branch_name):
    """Copy available baseline best checkpoints into the new run's ckpts/."""
    if "CHECKPOINT_PATHS" not in globals():
        return
    input_path = CHECKPOINT_PATHS.get(branch_name, "none")
    if not input_path or input_path == "none" or not os.path.exists(input_path):
        return

    abs_input = os.path.abspath(input_path)
    real_input = os.path.realpath(abs_input)
    dirs_to_check = set()
    files_to_check = set()
    for path in (abs_input, real_input):
        if os.path.isfile(path):
            files_to_check.add(path)
            dirs_to_check.add(os.path.dirname(path))
        elif os.path.isdir(path):
            dirs_to_check.add(path)

    best_files = set()
    patterns = ("best_fid_*.pth", "best_fid.pth", "best.pth", "bestfid*.pth",
                "best_*.pth", "*best*.pth")
    for directory in dirs_to_check:
        for pattern in patterns:
            best_files.update(
                path for path in glob.glob(os.path.join(directory, pattern))
                if os.path.isfile(path)
            )
    for path in files_to_check:
        if "best" in os.path.basename(path).lower():
            best_files.add(path)
    if not best_files and os.path.isfile(abs_input):
        best_files.add(abs_input)
    if not best_files:
        return

    runs_dir = os.path.join(MASTER_PATH, "HGGAN_test", branch_name, "runs")
    for _ in range(15):
        if os.path.isdir(runs_dir):
            run_dirs = [
                os.path.join(runs_dir, name) for name in os.listdir(runs_dir)
                if os.path.isdir(os.path.join(runs_dir, name))
            ]
            if run_dirs:
                target_dir = os.path.join(max(run_dirs, key=os.path.getmtime), "ckpts")
                os.makedirs(target_dir, exist_ok=True)
                for source_path in best_files:
                    target_path = os.path.join(target_dir, os.path.basename(source_path))
                    if not os.path.exists(target_path):
                        shutil.copy2(source_path, target_path)
                return
        time.sleep(1)


STATUS_ROOT = os.path.join(MASTER_PATH, "HGGAN_test")
JOB_SPECS = []
STATUS_GPU0_PATH = os.path.join(STATUS_ROOT, "job_gpu0.status.json")
STATUS_GPU1_PATH = os.path.join(STATUS_ROOT, "job_gpu1.status.json")
PID_GPU0_PATH = os.path.join(STATUS_ROOT, "job_gpu0.pid")
PID_GPU1_PATH = os.path.join(STATUS_ROOT, "job_gpu1.pid")


def launch_job(gpu_id, branch_name, config_path, stage, log_path, status_path, pid_path):
    if branch_name == "none":
        return
    for stale_path in (status_path, pid_path):
        try:
            os.remove(stale_path)
        except FileNotFoundError:
            pass
    branch_dir = os.path.join(MASTER_PATH, "HGGAN_test", branch_name)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    job_name = f"{branch_name}-{stage}"
    command = (
        f"cd {shlex.quote(branch_dir)} && "
        f"env HGGAN_STATUS_PATH={shlex.quote(status_path)} "
        f"HGGAN_JOB_NAME={shlex.quote(job_name)} PYTHONUNBUFFERED=1 "
        f"CUDA_VISIBLE_DEVICES={gpu_id} nohup {shlex.quote(sys.executable)} train.py "
        f"--config {shlex.quote(config_path)} > {shlex.quote(log_path)} 2>&1 & "
        f"echo $! > {shlex.quote(pid_path)}"
    )
    if os.system(command) != 0:
        raise RuntimeError(f"Failed to launch {branch_name} on GPU {gpu_id}")
    JOB_SPECS.append({
        'gpu': gpu_id, 'branch': branch_name, 'stage': stage,
        'log': log_path, 'status': status_path, 'pid': pid_path,
    })
    print(f"GPU {gpu_id}: {branch_name} ({config_path})")
    copy_baseline_best_to_run_folder(branch_name)


if RUN_GAN_JOBS:
    launch_job(0, BG_BRANCH_GPU0, BG_CONFIG_GPU0, "GAN",
               os.path.join(STATUS_ROOT, "train_gpu0.log"), STATUS_GPU0_PATH, PID_GPU0_PATH)
    launch_job(1, BG_BRANCH_GPU1, BG_CONFIG_GPU1, "GAN",
               os.path.join(STATUS_ROOT, "train_gpu1.log"), STATUS_GPU1_PATH, PID_GPU1_PATH)


In [ ]:
# ========================================================
# STEP 4: OCR TEACHER PRETRAINING
# GPU 0 -> MAIN OCR32, GPU 1 -> DEV OCR64
# ========================================================
import os
import shlex
import sys


STATUS_ROOT = os.path.join(MASTER_PATH, "HGGAN_test")
if "JOB_SPECS" not in globals():
    JOB_SPECS = []


def launch_training(gpu, branch, config):
    if branch == "none":
        return
    stem = os.path.splitext(os.path.basename(config))[0]
    log_path = os.path.join(STATUS_ROOT, f"train_gpu{gpu}.log")
    status_path = os.path.join(STATUS_ROOT, f"job_gpu{gpu}.status.json")
    pid_path = os.path.join(STATUS_ROOT, f"job_gpu{gpu}.pid")
    for path in (status_path, pid_path):
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
    branch_dir = os.path.join(MASTER_PATH, "HGGAN_test", branch)
    command = (
        f"cd {shlex.quote(branch_dir)} && "
        f"env HGGAN_STATUS_PATH={shlex.quote(status_path)} "
        f"HGGAN_JOB_NAME={shlex.quote(branch + '-' + stem)} PYTHONUNBUFFERED=1 "
        f"CUDA_VISIBLE_DEVICES={gpu} nohup {shlex.quote(sys.executable)} train.py "
        f"--config {shlex.quote(config)} > {shlex.quote(log_path)} 2>&1 & "
        f"echo $! > {shlex.quote(pid_path)}"
    )
    if os.system(command) != 0:
        raise RuntimeError(f"Could not launch {branch} on GPU {gpu}")
    JOB_SPECS.append({'gpu': gpu, 'branch': branch, 'log': log_path, 'status': status_path, 'pid': pid_path})
    print(f"✅ GPU {gpu}: {branch} ({config})")


if RUN_TEACHER_PRETRAIN:
    print(f"🚀 Launching {GPU0_CONFIG} and {GPU1_CONFIG}...")
    launch_training(0, GPU0_BRANCH, GPU0_CONFIG)
    launch_training(1, GPU1_BRANCH, GPU1_CONFIG)
else:
    print("ℹ️ Teacher pretraining disabled. Set RUN_TEACHER_PRETRAIN = True to launch.")


## 📊 Step 5: Real-Time Log Monitor
The monitor streams the selected logs and exits automatically when their status is terminal or when `TIME_LIMIT` hours have elapsed.

In [ ]:
import json
import os
import time

TERMINAL_STATES = {"completed", "failed", "interrupted"}
job_specs = list(globals().get("JOB_SPECS", []) or [])
if not job_specs:
    root = os.path.join(MASTER_PATH, "HGGAN_test")
    if globals().get("RUN_GAN_JOBS", False):
        for gpu, branch in ((0, globals().get("BG_BRANCH_GPU0", "none")),
                            (1, globals().get("BG_BRANCH_GPU1", "none"))):
            if branch != "none":
                job_specs.append({
                    "gpu": gpu,
                    "branch": branch,
                    "stage": "GAN",
                    "log": os.path.join(root, f"train_gpu{gpu}.log"),
                    "status": os.path.join(root, f"job_gpu{gpu}.status.json"),
                    "pid": os.path.join(root, f"job_gpu{gpu}.pid"),
                })
    if globals().get("RUN_TEACHER_PRETRAIN", False):
        for gpu, branch in ((0, globals().get("GPU0_BRANCH", "none")),
                            (1, globals().get("GPU1_BRANCH", "none"))):
            if branch != "none":
                job_specs.append({
                    "gpu": gpu,
                    "branch": branch,
                    "stage": "OCR",
                    "log": os.path.join(root, f"train_gpu{gpu}.log"),
                    "status": os.path.join(root, f"job_gpu{gpu}.status.json"),
                    "pid": os.path.join(root, f"job_gpu{gpu}.pid"),
                })

active = [spec for spec in job_specs if spec.get("branch") != "none"]
time_limit_hours = globals().get("TIME_LIMIT", None)
if time_limit_hours is not None and float(time_limit_hours) > 0:
    time_limit_hours = float(time_limit_hours)
    time_limit_sec = time_limit_hours * 3600.0
    print(f"👀 Monitoring jobs (time limit: {time_limit_hours}h); this cell exits when all jobs finish or time limit expires.")
else:
    time_limit_hours = None
    time_limit_sec = None
    print("👀 Monitoring jobs; this cell exits when all jobs finish.")

if not active:
    print("ℹ️ No jobs selected.")
    TRAINING_MONITOR_FINISHED = True
handles = {spec["gpu"]: open(spec["log"], "r", encoding="utf-8", errors="replace") if os.path.exists(spec["log"]) else None for spec in active}
monitor_start_time = time.time()


def status(spec):
    try:
        with open(spec["status"], encoding="utf-8") as handle:
            value = json.load(handle)
        return value if isinstance(value, dict) else {}
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        return {}


def alive(spec):
    try:
        with open(spec["pid"], encoding="utf-8") as handle:
            os.kill(int(handle.read().strip()), 0)
        return True
    except (FileNotFoundError, ValueError, ProcessLookupError, PermissionError, OSError):
        return False


def drain(spec):
    handle = handles.get(spec["gpu"])
    if handle is None and os.path.exists(spec["log"]):
        handle = open(spec["log"], "r", encoding="utf-8", errors="replace")
        handles[spec["gpu"]] = handle
    if handle is None:
        return False
    emitted = False
    for line in iter(handle.readline, ""):
        print(f"[{spec['branch']}] {line.rstrip()}")
        emitted = True
    return emitted


try:
    while active:
        # Check if runtime exceeded TIME_LIMIT
        if time_limit_sec is not None and (time.time() - monitor_start_time) >= time_limit_sec:
            for spec in active:
                drain(spec)
            elapsed_hrs = (time.time() - monitor_start_time) / 3600.0
            print(f"\n⏱️ TIME_LIMIT reached ({elapsed_hrs:.2f}h >= {time_limit_hours}h).")
            print("🛑 Stopping monitor print cell and terminating background processes for usage control...")
            pids_to_kill = []
            for spec in active:
                try:
                    with open(spec["pid"], encoding="utf-8") as handle:
                        pid = int(handle.read().strip())
                    pids_to_kill.append((spec, pid))
                    # Also find child worker processes (e.g. torch DataLoader workers)
                    child_pids = []
                    try:
                        with open(f"/proc/{pid}/task/{pid}/children", "r") as f:
                            child_pids = [int(p) for p in f.read().split()]
                    except Exception:
                        pass
                    for target_pid in child_pids + [pid]:
                        try:
                            os.kill(target_pid, 15)  # SIGTERM
                        except Exception:
                            pass
                    print(f"  ➡️ Sent SIGTERM to GPU {spec['gpu']} [{spec['branch']}] (PID {pid}, children: {child_pids})")
                except Exception:
                    pass

            # Grace period for processes to exit cleanly and flush logs
            time.sleep(2.0)
            for spec, pid in pids_to_kill:
                if alive(spec):
                    try:
                        os.kill(pid, 9)  # Force SIGKILL if still alive
                        print(f"  ⚡ Force-killed GPU {spec['gpu']} (PID {pid}) with SIGKILL")
                    except Exception:
                        pass
                drain(spec)

            TRAINING_MONITOR_TIMED_OUT = True
            break

        snapshot = []
        output = False
        done = True
        for spec in active:
            output = drain(spec) or output
            data = status(spec)
            finished = data.get("status") in TERMINAL_STATES
            if not finished and data and data.get("status") == "running" and not alive(spec):
                data = {**data, "status": "failed", "error": "process exited before completion"}
                finished = True
            if not finished and not data and os.path.exists(spec["pid"]) and not alive(spec):
                data = {"status": "failed", "error": "process exited before publishing status"}
                finished = True
            done = done and finished
            snapshot.append((spec, data))
        if done:
            for spec in active:
                drain(spec)
            print("\n✅ All jobs finished. RESULTS:")
            for spec, data in snapshot:
                print(f"  GPU {spec['gpu']} [{spec['branch']}] {data.get('status', 'unknown')}: {data.get('results_path', 'not published')}")
            TRAINING_MONITOR_FINISHED = True
            break
        if not output:
            time.sleep(0.5)
finally:
    for handle in handles.values():
        if handle is not None:
            handle.close()


## 🔁 Step 5.5: Teacher Output Check (optional)
After the OCR32/OCR64 jobs finish, inspect their `RESULTS.txt` paths from the monitor. Do not copy teachers between resolutions. Launch matching W+B configs only after their corresponding OCR export files exist.

In [ ]:
# Optional completion note
if globals().get("TRAINING_MONITOR_FINISHED", False):
    print("✅ Finished. Each model printed its RESULTS table and the monitor exited.")
elif globals().get("TRAINING_MONITOR_TIMED_OUT", False):
    print("⏱️ Monitor stopped because TIME_LIMIT was reached (session auto-stopping).")


## 🛑 Step 6: Terminate Background Training Processes (Customizable)
Terminates active training background jobs. You can customize the target to kill training only on **GPU 0**, **GPU 1**, or **both**.

In [ ]:
# # ========================================================
# # CONFIGURATION: Choose which GPU training process to terminate
# # Choices: "none", "GPU0", "GPU1", or "BOTH"
# # ========================================================
# TERMINATE_TARGET = "BOTH"

# import os, signal, subprocess

# print(f"🛑 Terminating background processes for target: {TERMINATE_TARGET}...")

# def terminate_gpu(gpu):
#     pids = set()
#     # 1. Match PIDs by environment variable
#     for pid in [p for p in os.listdir('/proc') if p.isdigit()]:
#         try:
#             with open(f"/proc/{pid}/environ", "rb") as f:
#                 if f"CUDA_VISIBLE_DEVICES={gpu}".encode() in f.read():
#                     pids.add(int(pid))
#         except Exception: pass
#     # 2. Match PIDs by nvidia-smi process table
#     try:
#         for line in subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL).decode().splitlines():
#             parts = line.replace('|', ' ').split()
#             if len(parts) >= 3 and parts[0] == gpu:
#                 for i, p in enumerate(parts):
#                     if p in ['C', 'G', 'C+G'] and parts[i-1].isdigit():
#                         pids.add(int(parts[i-1]))
#     except Exception: pass

#     # 3. Safely terminate identified python/train.py PIDs
#     killed = 0
#     for pid in pids:
#         if pid == os.getpid(): continue
#         try:
#             with open(f"/proc/{pid}/cmdline", "rb") as f:
#                 cmd = f.read()
#             if b"python" in cmd or b"train.py" in cmd:
#                 os.kill(pid, signal.SIGTERM)
#                 print(f"  ➡️ Killed PID {pid}")
#                 killed += 1
#         except Exception: pass
#     return killed

# if TERMINATE_TARGET in ["GPU0", "BOTH"]:
#     print("Checking GPU 0...")
#     n = terminate_gpu("0")
#     print(f"✅ Terminated {n} process(es) on GPU 0.")
# if TERMINATE_TARGET in ["GPU1", "BOTH"]:
#     print("Checking GPU 1...")
#     n = terminate_gpu("1")
#     print(f"✅ Terminated {n} process(es) on GPU 1.")
# if TERMINATE_TARGET == "none":
#     print("ℹ️ No termination targeted (set to 'none').")
# elif TERMINATE_TARGET not in ["GPU0", "GPU1", "BOTH", "none"]:
#     print("⚠️ Invalid target configuration!")

In [ ]:
# ========================================================
# DEV FID/KID/HWD BASELINE BENCHMARK (x32 checkpoints -> x64 IAM)
# Runs SpiS-GAN first, then FW-GAN, sequentially on one GPU.
# Attach checkpoints as a Kaggle dataset or set the two overrides below.
# ========================================================
import os
import subprocess
import sys

RUN_BASELINE_BENCHMARK = False

BENCHMARK_DEVICE = "cuda:0"
BENCHMARK_BATCH_SIZE = 8
BENCHMARK_NUM_WORKERS = 4
BENCHMARK_MAX_BATCHES = 0      # 0 = the complete IAM x64 test split
BENCHMARK_SEED = 123456
BENCHMARK_METRICS = "fid,kid,hwd"
SPIS_CHECKPOINT = ""           # Empty: auto-find bestIAM.pth under /kaggle/input
FW_CHECKPOINT = ""             # Empty: auto-find FW-GAN.pth under /kaggle/input

if not RUN_BASELINE_BENCHMARK:
    print("ℹ️ Baseline benchmark skipped (set RUN_BASELINE_BENCHMARK = True to run it).")
else:
    benchmark_runner = os.path.join(MASTER_PATH, "HGGAN_test", "dev", "tools", "run_external_benchmarks.py")
    command = [
        sys.executable, benchmark_runner,
        "--workspace", MASTER_PATH,
        "--device", BENCHMARK_DEVICE,
        "--batch-size", str(BENCHMARK_BATCH_SIZE),
        "--num-workers", str(BENCHMARK_NUM_WORKERS),
        "--max-batches", str(BENCHMARK_MAX_BATCHES),
        "--seed", str(BENCHMARK_SEED),
        "--metrics", BENCHMARK_METRICS,
    ]
    if SPIS_CHECKPOINT:
        command.extend(["--spis-checkpoint", SPIS_CHECKPOINT])
    if FW_CHECKPOINT:
        command.extend(["--fw-checkpoint", FW_CHECKPOINT])

    print("Running matched DEV validation: SpiS-GAN -> FW-GAN")
    print("Protocol: x64 reference -> native x32 style encoder/G -> bilinear x64 -> identical DEV evaluators")
    subprocess.run(command, check=True)
    print(os.path.join(MASTER_PATH, "baseline_dev_metric_benchmark", "spis_fw_dev_all_metrics.csv"))


In [ ]:
# ========================================================
# ARCHIVED LOCAL NORMAL-RUN UTILITIES
# DO NOT EXECUTE: every original line below is commented.
# Source: one_for_all_local_HTG.ipynb cells 14, 18, 20, 22
# To restore it, copy this cell to a new code cell and remove one leading "#".
# ========================================================
# # --- local cell 14: GPU status ---
# !nvidia-smi
#
# # --- local cell 18: package checkpoints ---
# import zipfile
# import os
# import glob
# import time
# from IPython.display import FileLink
#
# zip_filename = os.path.join(MASTER_PATH, "HGGAN_test", "all_checkpoints.zip")
# print("📦 Gathering and packaging model checkpoints (best & latest) from each branch...")
#
# count = 0
# with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
#     for branch in ["main", "dev", "classic_optimized", "alpha"]:
#         runs_dir = os.path.join(MASTER_PATH, "HGGAN_test", branch, "runs")
#         if os.path.exists(runs_dir):
#             pth_files = glob.glob(os.path.join(runs_dir, "**", "*.pth"), recursive=True)
#             pth_files = [f for f in pth_files if not os.path.basename(f).startswith('.tmp_')]
#             if pth_files:
#                 for pth in sorted(set(pth_files), key=os.path.getmtime, reverse=True):
#                     arcname = os.path.relpath(pth, os.path.join(MASTER_PATH, "HGGAN_test"))
#                     zipf.write(pth, arcname)
#                     mtime_str = time.ctime(os.path.getmtime(pth))
#                     print(f"   📦 Added: {arcname} (Modified: {mtime_str})")
#                     count += 1
#
# if count > 0:
#     print(f"\n🎉 Successfully packaged {count} checkpoint file(s) into: {zip_filename}")
#     print("🔗 Click the link below to download your checkpoints:")
#     display(FileLink(zip_filename))
# else:
#     print("⚠️ No checkpoints found to package.")
#
#
# # --- local cell 20: commit and push branches ---
# # ========================================================
# # CONFIGURATION: Commit & Push settings
# # Choices: "none", "all", "main", "dev", "classic_optimized", "alpha"
# # ========================================================
# COMMIT_BRANCH = "all"
# COMMIT_MESSAGE = "Update model configuration and training runs"
# MASTER_PATH = "/home/quq/machineLearning/HTG"
#
# import os
# import shutil
#
# if COMMIT_BRANCH == "none":
#     print("ℹ️ Skipping Git commit & push (set to 'none').")
# else:
#     ALL_BRANCHES = ["main", "dev", "classic_optimized", "alpha"]
#     branches_to_push = ALL_BRANCHES if COMMIT_BRANCH == "all" else [COMMIT_BRANCH]
#     
#     start_dir = os.getcwd()
#     
#     for branch in branches_to_push:
#         branch_path = os.path.join(MASTER_PATH, "HGGAN_test", branch)
#         if not os.path.exists(branch_path):
#             print(f"⚠️ Selected branch folder '{branch_path}' does not exist! Skipping.")
#             continue
#
#         print(f"🚀 Committing and force-pushing branch: {branch}...")
#         
#         # The -q hides the noisy directory printout on Kaggle
#         %cd -q {branch_path}
#
#         # Make sure Git credentials are set
#         !git config user.email "williamwrost617@gmail.com"
#         !git config user.name "rquq"
#
#         # 🧹 Refresh Git index to strictly enforce .gitignore
#         print("🧹 Refreshing Git index to drop cached ignores...")
#         !git rm -rf --cached . > /dev/null 2>&1
#         
#         # Stage all valid files
#         !git add .
#
#         # Check for changes, and commit only if there is something new
#         !git diff --cached --quiet || git commit -m "$COMMIT_MESSAGE"
#
#         # Force push changes to the exact branch on GitHub
#         !GIT_TERMINAL_PROMPT=0 git push --force origin HEAD:{branch}
#         print(f"✅ Finished branch: {branch}\n")
#         
#     %cd -q {start_dir}
#
# # --- local cell 22: external baseline benchmark ---
# # ========================================================
# # DEV FID/KID/HWD BASELINE BENCHMARK (x32 checkpoints -> x64 IAM)
# # Runs SpiS-GAN first, then FW-GAN, sequentially on one GPU.
# # Metrics: FID, KID, HWD, CMMD, CER, WER, IS-gen, IS-org, PSNR, MSSIM, WIER.
# # ========================================================
# import os
# import subprocess
# import sys
#
# BENCHMARK_DEVICE = "cuda:0"
# BENCHMARK_BATCH_SIZE = 4       # Full FP32; leaves headroom on the RTX 4060 8 GB
# BENCHMARK_NUM_WORKERS = 0      # Avoid duplicating the x64 HDF5 dataset in WSL workers
# BENCHMARK_MAX_BATCHES = 0      # 0 = the complete IAM x64 test split
# BENCHMARK_SEED = 123456
# BENCHMARK_METRICS = "fid,kid,hwd"
# SPIS_CHECKPOINT = ""         # Optional override; empty uses chkpoints/SpiS-GAN/bestIAM.pth
# FW_CHECKPOINT = ""           # Optional override; empty uses chkpoints/FW-GAN/FW-GAN.pth
#
# benchmark_runner = os.path.join(MASTER_PATH, "HGGAN_test", "dev", "tools", "run_external_benchmarks.py")
# command = [
#     sys.executable, benchmark_runner,
#     "--workspace", MASTER_PATH,
#     "--device", BENCHMARK_DEVICE,
#     "--batch-size", str(BENCHMARK_BATCH_SIZE),
#     "--num-workers", str(BENCHMARK_NUM_WORKERS),
#     "--max-batches", str(BENCHMARK_MAX_BATCHES),
#     "--seed", str(BENCHMARK_SEED),
#     "--metrics", BENCHMARK_METRICS,
# ]
# if SPIS_CHECKPOINT:
#     command.extend(["--spis-checkpoint", SPIS_CHECKPOINT])
# if FW_CHECKPOINT:
#     command.extend(["--fw-checkpoint", FW_CHECKPOINT])
#
# print("Running matched DEV validation: SpiS-GAN -> FW-GAN")
# print("Protocol: x64 reference -> native x32 style encoder/G -> bilinear x64 -> identical DEV evaluators")
# subprocess.run(command, check=True)
# print(os.path.join(MASTER_PATH, "baseline_dev_metric_benchmark", "spis_fw_dev_all_metrics.csv"))
